# SIH26184 — Cybercrime Cash-Withdrawal Location Forecasting
## Interactive Prototype Demonstration Notebook

> **Disclaimer:** This prototype uses synthetic data for hackathon prototype/demo purposes. Model performance demonstrates technical workflow and experimental behavior only.

In [ ]:
import os
import sys
import pandas as pd

# Ensure project root is in sys.path
sys.path.append("..")

from ml.data_loader import load_raw_data
from ml.predict import predict_locations
from ml.explain import explain_candidate_prediction
from ml.simulate import simulate_transaction

print("All modules imported successfully!")

### 1. Dataset Inspection & Summary

In [ ]:
cases_df, accounts_df, transactions_df, locations_df, withdrawals_df = load_raw_data()
print(f"Cases:        {cases_df.shape}")
print(f"Accounts:     {accounts_df.shape}")
print(f"Transactions: {transactions_df.shape}")
print(f"Locations:    {locations_df.shape}")
print(f"Withdrawals:  {withdrawals_df.shape}")

locations_df[['location_id', 'city', 'district', 'state', 'atm_cluster_name']]

### 2. Model Performance Benchmark

In [ ]:
comp_df = pd.read_csv("../results/model_comparison.csv")
comp_df

### 3. Predict Probable Cash-Out Locations for an Active Case

In [ ]:
case_id = "C0001"
result = predict_locations(case_id)

print(f"Case: {result['case_id']}")
print(f"Cut-off Time (T): {result['prediction_time']}")
print(f"Estimated Cash-Out Window: {result['estimated_cashout_time_window']}")

pd.DataFrame(result["top_5"])[["rank", "location_id", "city", "atm_cluster_name", "score", "risk"]]

### 4. SHAP Feature Attribution for Top Prediction

In [ ]:
explanation = explain_candidate_prediction(case_id, target_rank=1)
print(f"Candidate: {explanation['city']} ({explanation['location_id']}) | Score: {explanation['model_score']} | Risk: {explanation['risk_tier']}")
print(f"Method: {explanation['explanation_method']}")

pd.DataFrame(explanation["contributing_factors"])[["feature_description", "feature_value", "attribution_score", "direction"]]

### 5. Dynamic Multi-Hop Simulation (New Evidence Arrival)

In [ ]:
new_evidence = {
    "sender_account": "A0001_5",
    "receiver_account": "A0001_9_NEW",
    "receiver_region": "Mumbai",
    "amount": 25000,
    "timestamp": "2026-06-13 05:45:00",
    "transaction_type": "Transfer"
}

sim = simulate_transaction(case_id, new_evidence)

print("=== BEFORE NEW EVIDENCE ===")
display(pd.DataFrame(sim["before_top_3"])[["rank", "city", "location_id", "score", "risk"]])

print("=== AFTER NEW EVIDENCE ===")
display(pd.DataFrame(sim["after_top_3"])[["rank", "city", "location_id", "score", "risk"]])

print("=== RANK SHIFTS ===")
display(pd.DataFrame(sim["top_shifts"])[["city", "location_id", "before_rank", "after_rank", "rank_change", "before_score", "after_score", "score_change"]])